# Claude Study 01 — Agent Architecture & the Agent SDK

**Companion notebook to** `Claude_Study_01_Agent_Architecture_and_the_Agent_SDK.md`.

Every section below states three things before any code runs:

- **Concept** — the idea from the study doc this section demonstrates.
- **SDK surface** — the exact API/parameter you are learning.
- **What this proves** — the claim the output verifies.

| § | Concept | SDK surface |
|---|---|---|
| 0 | Environment works | `anthropic.Anthropic()`, `messages.create()` |
| 1 | Workflow vs agent | `stop_reason` |
| 2 | The agentic loop, by hand | `tools=[...]`, `tool_use` / `tool_result` |
| 3 | Tool stuffing is a per-call tax | `response.usage.input_tokens` |
| 4 | Reviewer / critic loop | two calls, structured judgment |
| 5 | Agent SDK | `claude_agent_sdk.query` + `ClaudeAgentOptions` |
| 6 | Hooks | `PreToolUse` / `PostToolUse`, `HookMatcher` |
| 7 | Managed Agents | `ant` CLI (reference only) |

**Cost:** sections 0–4 run on `claude-haiku-4-5` and cost well under **$0.01** total. Sections 5–6 use Sonnet and cost a few cents.

---
## 0. Setup — prove the environment works

**Concept.** Nothing below is meaningful until three things are true: the SDK is installed, a key is present, and a call reaches Anthropic and comes back.

**SDK surface.** `pip install anthropic` → `anthropic.Anthropic()` → `client.messages.create(...)`.

**What this proves.** That your key is live, and it introduces the three fields on *every* response you will read for the rest of the track: `content`, `stop_reason`, `usage`.

In [ ]:
# Step 1 of 3 — install the Anthropic Python SDK.
# -q keeps Colab's output short. This is the only install sections 0-4 need.
%pip install -q anthropic

### Step 2 of 3 — supply your API key

Pick whichever applies:

- **Colab (recommended).** Click the 🔑 key icon in the left sidebar → *Add new secret* → name it `ANTHROPIC_API_KEY`, paste the value, and toggle **Notebook access** on.
- **Local Jupyter.** Set `ANTHROPIC_API_KEY` in your shell before launching, or let the prompt below catch you.

The cell tries Colab Secrets, then the environment, then falls back to a hidden prompt. **Never type a key into a code cell** — it gets saved into the notebook file and then into git.

In [ ]:
import os, getpass

def load_key():
    try:                                        # 1) Colab Secrets
        from google.colab import userdata
        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass
    if os.environ.get("ANTHROPIC_API_KEY"):     # 2) already in the environment
        return os.environ["ANTHROPIC_API_KEY"]
    return getpass.getpass("ANTHROPIC_API_KEY: ")   # 3) hidden prompt

os.environ["ANTHROPIC_API_KEY"] = load_key()
print("Key loaded:", os.environ["ANTHROPIC_API_KEY"][:7] + "..." )

### Step 3 of 3 — one real call

`Anthropic()` reads `ANTHROPIC_API_KEY` from the environment on its own — you never pass the key as an argument.

Read the four printed lines carefully; they are the vocabulary for the whole track:

- **`content`** is a *list of typed blocks*, not a string. Today it holds one `text` block. In §2 the same list will hold a `tool_use` block. This is why you write `response.content[0].text` and not `response.text`.
- **`stop_reason`** is *why the model stopped*: `end_turn` (finished), `tool_use` (it wants a tool run), `max_tokens` (you truncated it), `refusal`. **This single field is the agent loop's condition** — §2 builds the loop on it.
- **`usage`** is what you were billed for, per call.
- **`model`** is what actually served the request.

In [ ]:
import anthropic

client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY from the environment

resp = client.messages.create(
    model="claude-haiku-4-5",           # cheapest current model; fine for a smoke test
    max_tokens=100,
    messages=[{"role": "user", "content": "Reply with exactly: OK"}],
)

print("model       :", resp.model)
print("content     :", resp.content)            # a LIST of typed blocks
print("text        :", resp.content[0].text)    # ...so you index into it
print("stop_reason :", resp.stop_reason)        # 'end_turn' = it finished on its own
print("usage       :", resp.usage)

✅ If you saw `OK` and `stop_reason: end_turn`, your environment is working and you can run everything below.

❌ `AuthenticationError` → the key is wrong or has no credit. `NameError` → run the cells above in order.

---
## 1. Workflow vs agent — the distinction, in code

**Concept.** *(Study doc, Parts 1–2.)* A **workflow** traverses transitions you enumerated in advance. An **agent** constructs a path you could not have drawn. The test: *can you list the transitions before it runs?*

**SDK surface.** None new — this section is deliberately about **when not to call the API**.

**What this proves.** That a large share of "agent" work is a `dict` lookup wearing a costume. Both cells below do the same job; one costs nothing and never varies.

In [ ]:
# WORKFLOW: transitions are enumerable, so this is a state machine. Write it as code.
# Zero tokens, zero latency, 100% reproducible, unit-testable.
ROUTES = {
    "billing":  "finance-queue",
    "outage":   "sre-pager",
    "password": "it-helpdesk",
}

def route(ticket_type: str) -> str:
    return ROUTES.get(ticket_type, "triage-queue")

for t in ["outage", "billing", "something-new"]:
    print(f"{t:<15} -> {route(t)}")

Now the part a `dict` genuinely cannot do: the input is free text, the categories are fuzzy, and judgment is required. **This** is what you buy a model for.

In [ ]:
TICKET = ("Checkout has been timing out for about 20 minutes. Customers see a spinner "
          "then a 502. We deployed a payments change this morning but rolled it back.")

resp = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=200,
    system="Classify the support ticket. Reply with exactly one line: <category> | <severity 1-5> | <one-sentence reason>",
    messages=[{"role": "user", "content": TICKET}],
)
print(resp.content[0].text)
print("\nstop_reason:", resp.stop_reason, "| input tokens:", resp.usage.input_tokens)

**The judgment call to carry into design reviews:** the first cell is correct, free and deterministic; the second is none of those, and is the only one that can read a paragraph it has never seen. Put the model at the one step that needs judgment — and leave the rest as code.

---
## 2. The agentic loop, built by hand

**Concept.** *(Study doc, Part 1.)* The "agentic loop" diagram is one `while` statement. This section removes the mystery by writing it out.

**SDK surface.** `tools=[{name, description, input_schema}]` · `stop_reason == "tool_use"` · the `tool_use` content block · the `tool_result` block you send back.

**What this proves.** That the loop is driven *entirely* by `stop_reason`, that **the API never runs your code** — it only asks you to — and that the conversation is **stateless**: you resend the whole history every turn.

In [ ]:
import json

# --- the tool, as the model sees it (a JSON schema) ---
TOOLS = [{
    "name": "get_deploy_info",
    "description": "Return the most recent deployment for a service.",
    "input_schema": {
        "type": "object",
        "properties": {"service": {"type": "string", "description": "e.g. 'checkout-api'"}},
        "required": ["service"],
    },
}]

# --- the tool, as YOUR code (the model can never execute this; it only asks you to) ---
FAKE_DEPLOYS = {
    "checkout-api": {"minutes_ago": 4320, "version": "v2.1.0"},
    "payments-svc": {"minutes_ago": 55,   "version": "v4.7.2"},
}

def get_deploy_info(service: str) -> dict:
    return FAKE_DEPLOYS.get(service, {"error": f"unknown service '{service}'"})

In [ ]:
messages = [{
    "role": "user",
    "content": "Checkout is failing. Did payments-svc deploy recently? Answer in one sentence.",
}]

MAX_TURNS = 5        # BOUND #1: never write `while True` against a paid API.
turn = 0

while turn < MAX_TURNS:
    turn += 1
    resp = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=500,
        tools=TOOLS,
        messages=messages,          # the FULL history, every turn - the API stores nothing
    )
    print(f"--- turn {turn} | stop_reason = {resp.stop_reason} ---")

    # The loop condition. This is the entire state machine.
    if resp.stop_reason != "tool_use":
        for block in resp.content:
            if block.type == "text":
                print("FINAL:", block.text)
        break

    # Claude asked for one or more tools. Append its turn verbatim...
    messages.append({"role": "assistant", "content": resp.content})

    # ...run each requested tool, and return every result in ONE user message.
    results = []
    for block in resp.content:
        if block.type == "tool_use":
            print(f"  tool call: {block.name}({json.dumps(block.input)})")
            output = get_deploy_info(**block.input)
            print(f"  result   : {output}")
            results.append({
                "type": "tool_result",
                "tool_use_id": block.id,      # must echo the id back
                "content": json.dumps(output),
            })
    messages.append({"role": "user", "content": results})
else:
    print(f"!! hit the {MAX_TURNS}-turn bound without finishing")

**Four things that output just demonstrated:**

1. **`stop_reason` is the loop.** `tool_use` → keep going; anything else → stop. There is no event, no callback — it is a field on a return value.
2. **You run the tool, not Anthropic.** The API returned a *request*. Your Python executed it. Everything an agent can do to your systems, it does through code you wrote.
3. **Stateless.** `messages` grows on every pass and is resent in full. Nothing is remembered server-side.
4. **Bounds are yours to add.** `MAX_TURNS` is the simplest of the three bounds you need in production — **turns, cost, wall-clock**. A `while True` here is an unbounded invoice.

> Everything in §5 and §7 is somebody else running *this exact loop* for you.

---
## 3. Tool stuffing is a tax on every call

**Concept.** *(Study doc, Part 3.)* Every tool you attach ships its **full JSON schema in every request**. Tool definitions are input tokens, paid per call — not once.

**SDK surface.** `response.usage.input_tokens`.

**What this proves.** The cost of attaching tools "just in case", measured rather than asserted — which is the argument for giving each subagent only the tools its job needs.

In [ ]:
def extra_tools(n):
    # Generate n plausible-looking tool definitions.
    return [{
        "name": f"lookup_system_{i}",
        "description": f"Look up detailed operational records in internal system {i}, "
                       f"including owner, status, last change, and dependencies.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query":  {"type": "string", "description": "the search query"},
                "limit":  {"type": "integer", "description": "max rows to return"},
                "since":  {"type": "string", "description": "ISO-8601 lower bound"},
            },
            "required": ["query"],
        },
    } for i in range(n)]

PROBE = [{"role": "user", "content": "Say OK."}]

def input_tokens_with(tools):
    r = client.messages.create(model="claude-haiku-4-5", max_tokens=10,
                               tools=tools, messages=PROBE)
    return r.usage.input_tokens

lean   = input_tokens_with(TOOLS)                    # 1 tool
stuffed = input_tokens_with(TOOLS + extra_tools(15))  # 1 + 15 tools

print(f"1  tool  -> {lean:>5} input tokens")
print(f"16 tools -> {stuffed:>5} input tokens")
print(f"overhead -> {stuffed - lean:>5} tokens on EVERY call ({stuffed/lean:.1f}x)")

Multiply that overhead by every turn of every conversation of every user. The same tokens also make the model choose from sixteen options instead of one — so trimming the tool list is a **cost** optimization and an **accuracy** optimization at once.

---
## 4. The reviewer / critic loop

**Concept.** *(Study doc, Part 3.)* Self-review fails: the thing that produced the error is biased toward not finding it. The production pattern is **worker emits decision + confidence + reasoning; a second agent reviews that triple.**

**SDK surface.** No new API — this is *architecture*. Two independent calls, the second seeing only the first's output.

**What this proves.** Run in two arms:

- **Arm 1** builds the mechanism — worker, then an independent reviewer.
- **Arm 2** is the one that proves anything: it hands the reviewer a verdict we **know is wrong** and checks whether it says so.

Why the second arm is mandatory: if the worker is right and the reviewer agrees, you have learned **nothing**. A reviewer that rubber-stamps everything produces exactly that output too. LLM reviewers skew agreeable — especially toward confident-sounding text — so agreement is also the expected output of a *broken* reviewer. **You can only measure a checker by giving it something to catch.**

In [ ]:
AMBIGUOUS = ("Checkout has been timing out for 20 minutes. Someone said it's probably the "
             "checkout-api deploy. Logs show payments-svc latency climbing since 09:05.")

FACTS = "Deploy history: checkout-api last deployed 4320 minutes ago. payments-svc last deployed 55 minutes ago."

def worker(ticket):
    r = client.messages.create(
        model="claude-haiku-4-5", max_tokens=400,
        system=("You are an incident triage agent. Using ONLY the facts given, output exactly three lines:\n"
                "CAUSE: <the service most likely at fault>\n"
                "CONFIDENCE: <0.0-1.0>\n"
                "REASON: <one sentence>"),
        messages=[{"role": "user", "content": f"{FACTS}\n\nTicket: {ticket}"}],
    )
    return r.content[0].text

def reviewer(ticket, verdict):
    r = client.messages.create(
        model="claude-haiku-4-5", max_tokens=400,
        system=("You are an independent reviewer. You did NOT write the verdict. "
                "Check it against the facts. Output exactly two lines:\n"
                "AGREE: yes|no\n"
                "CRITIQUE: <one sentence - name any claim unsupported by the facts>"),
        messages=[{"role": "user", "content": f"{FACTS}\n\nTicket: {ticket}\n\nVerdict under review:\n{verdict}"}],
    )
    return r.content[0].text

v = worker(AMBIGUOUS)
print("WORKER\n" + v)
print("\nREVIEWER\n" + reviewer(AMBIGUOUS, v))

**Arm 1, what to look for:** whether the worker followed the *stated* suspicion (`checkout-api`) or the *evidence* (`payments-svc`, deployed 55 minutes ago).

On this ticket the evidence is fairly one-sided, so the worker usually gets it right and the reviewer usually agrees. **That tells you nothing about the reviewer.** Hence arm 2.

In [ ]:
# ARM 2 — the control. Hand the reviewer a verdict we KNOW is wrong:
# it blames the service that was NOT recently deployed, states it with high
# confidence, and reads as completely plausible. A working reviewer says no.
CORRUPTED = """CAUSE: checkout-api
CONFIDENCE: 0.95
REASON: The checkout-api deploy this morning introduced the timeout."""

print("REVIEWER on the REAL verdict")
print(reviewer(AMBIGUOUS, v))
print("\n" + "=" * 60 + "\n")
print("REVIEWER on a KNOWN-WRONG verdict")
print(reviewer(AMBIGUOUS, CORRUPTED))

### Reading the result

| Reviewer said | Verdict on the reviewer |
|---|---|
| **yes** to the real one, **no** to the corrupted one | ✅ it discriminates — the pattern earns its 2× cost |
| **yes** to both | ❌ rubber stamp — you are paying double for theatre |
| **no** to both | ❌ reflexive critic — equally useless, just noisier |

**The transferable point, and it is bigger than this cell.** You cannot validate a checker by showing it one case it passes. You need **known-positives and known-negatives**, and you measure whether it *separates* them.

That is what an eval is. Which lands squarely on Part 0 of the study doc: the exam blueprint gives **Eval, Testing and Debugging 2.6%** — one question out of 53 — and this four-line control cell is why that weighting is wrong about production. Everyone building agents has an opinion about whether their reviewer works. Almost nobody has run the negative case.

The cost of the full pattern is roughly **2× tokens and 2× latency**. Spend it where correctness beats speed; skip it where it doesn't — and only after arm 2 says the reviewer can actually say no.

---
## 5. The Agent SDK — Claude Code as a library

**Concept.** *(Study doc, Parts 4–5.)* Sections 2–4 used the **Messages API**, where you wrote the loop and defined every tool. The **Agent SDK** is a *different product*: it hands you **Claude Code's** harness and its built-in tools — Read, Write, Edit, Bash, Glob, Grep, WebSearch — running on **your** machine.

**SDK surface.** `pip install claude-agent-sdk` · `query(prompt, options)` · `ClaudeAgentOptions(allowed_tools=..., permission_mode=...)` · `AssistantMessage` / `ResultMessage`.

**What this proves.** That `allowed_tools` *is* your security boundary, and that `query()` is an async generator you can watch the agent think through.

> ⚠️ **This section needs Node.js** — the Python package drives the Claude Code CLI. Colab has Node preinstalled; the install below takes ~1 minute. Sections 0–4 do not need any of this.

In [ ]:
# The Agent SDK is a SEPARATE package from `anthropic`, and needs the Claude Code CLI.
%pip install -q claude-agent-sdk
!npm install -g @anthropic-ai/claude-code --silent 2>/dev/null | tail -1
!claude --version

### What tools does *your* install actually have?

Don't trust a list from a document — tool names drift between versions. The SDK reports the real set at session start, in a `SystemMessage` with `subtype="init"`. Print it and you have ground truth for the version on this machine.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, SystemMessage

async def list_tools():
    async for m in query(
        prompt="Reply with the single word: ready",
        options=ClaudeAgentOptions(max_turns=1),   # no `tools=` -> the default set
    ):
        if isinstance(m, SystemMessage) and m.subtype == "init":
            d = m.data
            print("model :", d.get("model"))
            print("cwd   :", d.get("cwd"))
            print(f"\ntools ({len(d.get('tools', []))}):")
            for name in sorted(d.get("tools", [])):
                print("  ", name)
            break          # the init message is all we needed

await list_tools()

That list is what `tools=[...]` chooses from. The ones you will reach for most:

| Tool | Does |
|---|---|
| `Read` | read a file — also images, PDFs, notebooks |
| `Write` | create or overwrite a file |
| `Edit` | exact string replacement inside a file |
| `Glob` | find files by **name** pattern — `**/*.py` |
| `Grep` | search file **contents** (ripgrep) |
| `Bash` | run a shell command |
| `WebFetch` / `WebSearch` | fetch a URL / search the web |
| `Task` | spawn a subagent |

`Glob` vs `Grep` is the pair people mix up: **Glob matches filenames, Grep matches contents.**

**Permissions can be scoped, not just named.** `allowed_tools` and `disallowed_tools` accept a `Tool(specifier)` form, so pre-approval can be narrowed to an argument pattern:

```python
allowed_tools=["Read", "Bash(git status)", "Bash(git diff:*)"]
```

That pre-approves exactly those git commands and leaves every other `Bash` invocation to the permission path — which is only meaningful once `tools` has decided `Bash` exists at all.

In [ ]:
# Give the agent something real to look at.
!mkdir -p /content/demo

In [ ]:
%%writefile /content/demo/utils.py
def average(numbers):
    # BUG: ZeroDivisionError when `numbers` is empty
    return sum(numbers) / len(numbers)


In [ ]:
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ResultMessage

async def main():
    async for message in query(
        prompt="Review utils.py for bugs that would cause a crash. Describe the bug and the fix. Do not edit the file.",
        options=ClaudeAgentOptions(
            # `tools` = WHICH TOOLS EXIST for this agent. This is the real boundary:
            # Bash, Write and Edit are not in the set, so the agent cannot reach for them.
            tools=["Read", "Glob", "Grep"],
            # `allowed_tools` = WHICH TOOLS RUN WITHOUT ASKING. A permission rule,
            # NOT a restriction. Anything here is auto-approved; anything omitted
            # still exists and goes through the permission path.
            allowed_tools=["Read", "Glob", "Grep"],
            cwd="/content/demo",
            max_turns=8,            # built-in bound: turns
            max_budget_usd=0.50,    # built-in bound: spend
        ),
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if hasattr(block, "text"):
                    print(block.text)                       # Claude's reasoning
                elif hasattr(block, "name"):
                    print(f"[tool] {block.name}")           # a tool being called
        elif isinstance(message, ResultMessage):
            print(f"\n[done] {message.subtype}")

await main()        # Jupyter already runs an event loop, so `await` directly

### ⚠️ `allowed_tools` is not the blast radius — `tools` is

An earlier version of this notebook passed only `allowed_tools=["Read", "Glob", "Grep"]` and called it a read-only agent. **Running it printed `[tool] Bash`.** The claim was wrong, and the run disproved it.

`ClaudeAgentOptions` carries three separate tool settings, and they do different jobs:

| Option | Question it answers | Effect |
|---|---|---|
| **`tools`** | *which tools exist at all?* | **the availability set — the real boundary** |
| `allowed_tools` | *which run without asking?* | auto-approve (a permission rule) |
| `disallowed_tools` | *which are refused outright?* | auto-deny |

`allowed_tools` is a **pre-approval list, not an allowlist.** Omitting `Bash` from it never removed Bash — it only meant Bash would go through the permission path, and in a non-interactive run it still executed. The same split exists on the CLI: `--tools` sets what is available, `--allowedTools` sets what is pre-approved.

Verify it yourself in the next cell.

**Also worth noting above:** `max_turns` and `max_budget_usd` are **built into the SDK**. In §2 you had to write `MAX_TURNS` by hand because the raw Messages API gives you nothing — here, two of the three bounds (turns, cost) are parameters. Wall-clock is still yours.

In [ ]:
# PROOF. Same prompt, same read-only `tools` set, but now we actively invite
# the agent to use a shell. Collect every tool it manages to call.
used = []

async def probe():
    async for message in query(
        prompt="Use bash to list the files here, then read utils.py and report its bug.",
        options=ClaudeAgentOptions(
            tools=["Read", "Glob", "Grep"],      # Bash is NOT in the available set
            allowed_tools=["Read", "Glob", "Grep"],
            cwd="/content/demo",
            max_turns=8,
            max_budget_usd=0.50,
        ),
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if hasattr(block, "name"):
                    used.append(block.name)

await probe()
print("tools actually used:", sorted(set(used)) or "none")
print("Bash reached?       :", "Bash" in used)     # expect False

If `Bash reached? False` — **`tools` held**, even when the prompt asked for a shell directly. That is a boundary. `allowed_tools` alone was not.

The lesson generalizes past this SDK: **a setting named "allowed" is not automatically a restriction.** Read what the parameter governs — availability or approval — before you treat it as a control. And then do what you just did: write the prompt that tries to break it, and look at what actually ran.

---
## 6. Hooks — the deterministic control plane

**Concept.** *(Study doc, Part 6.)* A system prompt saying "never touch production config" is a *request* to a probabilistic system. A `PreToolUse` hook that inspects the arguments and refuses is a *guarantee*.

**SDK surface.** `ClaudeAgentOptions(hooks={"PreToolUse": [HookMatcher(matcher=..., hooks=[fn])], ...})`.

**What this proves.** That you can run **your code** at a known moment in the agent's life — before and after every tool — and that `matcher` is a regex you narrow in production (`"Edit|Write"`), not the `".*"` of the demo.

In [ ]:
from claude_agent_sdk import HookMatcher

AUDIT = []     # in production this is your log sink

async def pre_tool_hook(input_data, tool_use_id, context):
    tool = input_data.get("tool_name", "unknown")
    AUDIT.append(("PRE", tool, tool_use_id))
    print(f"[PRE ] {tool:<8} | {tool_use_id}")
    return {}          # {} = allow. This is where a real gate would deny.

async def post_tool_hook(input_data, tool_use_id, context):
    tool = input_data.get("tool_name", "unknown")
    AUDIT.append(("POST", tool, tool_use_id))
    print(f"[POST] {tool:<8} | {tool_use_id}")
    return {}

async def main_hooked():
    async for message in query(
        prompt="List the Python files here and summarize what utils.py does. Do not edit anything.",
        options=ClaudeAgentOptions(
            tools=["Read", "Glob"],            # availability: the real boundary
            allowed_tools=["Read", "Glob"],    # pre-approval: no prompting
            cwd="/content/demo",
            hooks={
                "PreToolUse":  [HookMatcher(matcher=".*", hooks=[pre_tool_hook])],
                "PostToolUse": [HookMatcher(matcher=".*", hooks=[post_tool_hook])],
            },
        ),
    ):
        if isinstance(message, ResultMessage):
            print(f"\n[done] {message.subtype}")

await main_hooked()
print(f"\nAudit trail: {len(AUDIT)} hook events captured")

**Why this is the important cell in the notebook.** Every compliance requirement you will be handed — audit logging, PII redaction, path allow-lists, spend caps — belongs **here**, not in the prompt. The prompt is documentation; the hook is the control.

`matcher=".*"` catches everything, which is the demo setting. In production you narrow it to the tools that can do damage:

```python
hooks={"PreToolUse": [HookMatcher(matcher="Edit|Write|Bash", hooks=[deny_outside_sandbox])]}
```

---
## 7. Managed Agents — reference only

**Concept.** *(Study doc, Part 7.)* The one option where **Anthropic runs the loop *and* hosts the sandbox**. Not runnable from Colab — it needs the `ant` CLI, a browser login, and server-side resources that cost money — so this section is a map, not an exercise.

**Five objects, created in this order:**

| Object | What it is |
|---|---|
| **Agent** | model + system prompt + tools + MCP + skills — stored and versioned |
| **Environment** | the sandbox *machine*: cloud or self-hosted, networking, packages (**not** env vars) |
| **Session** | one running instance performing one task, emitting events |
| **Deployment** | agent + environment + credentials bound to a trigger: `Manual` or `Schedule` |
| **Credential vault** | secrets the run may use (MCP servers, other tools) |

```bash
ant auth login                      # separate login from Claude Code
ant apply coding-assistant.md       # create the Agent      -> writes claude-lock.json
ant apply environment.yaml          # create the Environment
ant beta:sessions create \          # create the Session (needs BOTH ids)
  --agent "$AGENT_ID" --environment-id "$ENVIRONMENT_ID" \
  --title "Quickstart session" --transform id --raw-output
```

**Two field notes worth more than the commands:**

1. **The quickstart's steps don't chain.** Follow the CLI tabs and you get an agent, an environment and a session — then the streaming example hands you SDK code that *creates its own session*. Use `retrieve`, not `create`, to continue from where the CLI left you.
2. **The real payoff isn't the hosting — it's the trace.** The Console shows every tool call, result, thinking step, timing and per-session cost. That is agent observability you would otherwise build yourself.

---
## What you just proved

| § | Claim | Evidence |
|---|---|---|
| 0 | The environment works | a live call returning `end_turn` |
| 1 | Most "agent" work is a dict | same job, zero tokens |
| 2 | The loop *is* `stop_reason` | the turn-by-turn trace |
| 2 | The API never runs your code | your Python executed the tool |
| 3 | Tools are a per-call tax | measured input-token overhead |
| 4 | A reviewer is only worth its cost if it can say **no** | arm 2: the known-wrong verdict |
| 5 | `tools` is the blast radius, `allowed_tools` is not | the probe: Bash never reached |
| 6 | Hooks enforce; prompts request | an audit trail captured in code |

**The two questions this whole lesson reduces to:**

1. **Is this an agent at all?** Can you enumerate the transitions in advance? If yes, it's a workflow — build it in code and call an agent from the one step that needs judgment.
2. **Whose harness, whose machine?** Manual loop (§2: neither) → Agent SDK (§5: Claude Code's harness, your machine) → Managed Agents (§7: both theirs).

**Next:** Claude Study 02 — model selection and optimization (the 16.8% domain), where `usage` from §0 becomes a cost model.